In [1]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

/Users/diego/.pyenv/versions/langchain/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [3]:
embd=embeddings.embed_documents(texts=["What is the capital of France?","abadi es negrito"])
print(len(embd), type(embd), len(embd[0]), type(embd[0]))

2 <class 'list'> 3072 <class 'list'>


In [4]:
file_path = "./documents/big_chunk.md"

In [20]:
file_path = "./documents/big_chunk.md"
with open(file_path, "r") as f:
    md_text = f.read()

print(md_text[:50])

# NETWORKING

## 1. BGP Session Down: Neighbor Pee


In [26]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document

headers_to_split_on = [
    ("#", "Category"),    # Captures NETWORKING, SECURITY, HARDWARE
    ("##", "Title"),      # Captures the specific issue name
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, 
    strip_headers=False # Keep headers in the text so the LLM sees them
)

md_header_splits = markdown_splitter.split_text(md_text)

final_docs = []
for doc in md_header_splits:
    # Logic to extract Zabbix ID from the text and move it to metadata
    content = doc.page_content
    if "Zabbix Error ID:" in content:
        z_id = content.split(sep="**Zabbix Error ID:**")[1].split("\n")[0].strip(" `")
        doc.metadata["zabbix_id"] = z_id
    final_docs.append(doc)


In [34]:
md_header_splits[2]

Document(metadata={'Category': 'NETWORKING', 'Title': '3. Anycast Routing Latency Spike', 'zabbix_id': 'NET-LAT-ANY-009'}, page_content='## 3. Anycast Routing Latency Spike\n**Zabbix Error ID:** `NET-LAT-ANY-009`  \n**Description:**\nInternal Anycast VIPs are experiencing >50ms latency. This typically indicates a "Routing Flap" where traffic is being routed to a geographically distant PoP instead of the local node.  \n**Step-by-step Solution:**\n1. **Trace Route:** Run `mtr -rn [VIP_Address]` from the affected client zone to see the path.\n2. **Identify Node:** Check which node is currently "winning" the BGP advertisement for that VIP.\n3. **Verify Health Check:** Check if the local health-checking daemon (e.g., Bird or ExaBGP) has withdrawn the route due to a local service failure.\n4. **Force Re-convergence:** Restart the BGP daemon on the local node to re-announce the route with a lower MED or shorter AS-Path.\n5. **Review Quagga/Bird Logs:** Look for "Route flap dampened" messages.

In [67]:
vectorstore = Chroma.from_documents(documents=md_header_splits, 
                                    embedding=embeddings,
                                    persist_directory="./chroma_db")
retriever = vectorstore.as_retriever(search_kwargs={"k":2})
retriever = vectorstore.as_retriever(search_kwargs={"filter": {"Category": "HARDWARE"}})

In [66]:
search_results = retriever.invoke("SSD S.M.A.R.T. Failure Prediction",)
print(len(search_results))


4
